# Transformer Architecture: English to Portuguese Translation

## Overview

This tutorial demonstrates a **Transformer** model for sequence-to-sequence translation. Unlike RNNs/LSTMs that process sequences sequentially, Transformers use **self-attention** to process all positions in parallel.

### Key Advantages of Transformers:
1. **Parallelization**: All positions processed simultaneously (faster training)
2. **Long-range Dependencies**: Attention directly connects distant positions without intermediate steps
3. **Scalability**: Better scaling properties for larger datasets
4. **State-of-the-art**: Foundation for GPT, BERT, and modern NLP models

### Architecture Overview:
```
Input → Embedding + Positional Encoding → Encoder (N layers) ↘
                                                          → Output
Target → Embedding + Positional Encoding → Decoder (N layers) ↗
```

Each layer contains:
- **Multi-Head Self-Attention**: Multiple attention "heads" looking at different representation subspaces
- **Feed-Forward Network**: Position-wise dense layers
- **Layer Normalization**: Stabilizes training
- **Residual Connections**: Enables deep networks

## Step 1: Import Libraries and Configure Parameters

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Dense, Layer, LayerNormalization, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

/opt/anaconda3/envs/tensorflow/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


TensorFlow version: 2.21.0
GPU Available: []


### Hyperparameters Explained

- **EMBEDDING_DIM** (d_model): Dimension of the model (query/key/value dimensions)
- **NUM_HEADS**: Number of attention heads (model dimension must be divisible by this)
- **FF_DIM**: Hidden dimension of feed-forward network (typically 4x embedding dimension)
- **NUM_LAYERS**: Depth of encoder/decoder stacks
- **DROPOUT_RATE**: Regularization to prevent overfitting

In [2]:
# ==========================================
# CONFIGURATION
# ==========================================

FILE_PATH = "por.txt"
NUM_EXAMPLES = 5000      # Number of training examples
BATCH_SIZE = 64          # Sequences per batch
EMBEDDING_DIM = 128      # Model dimension (d_model)
NUM_HEADS = 4            # Number of attention heads
FF_DIM = 512             # Feed-forward hidden dimension
NUM_LAYERS = 2           # Encoder/Decoder layer count
DROPOUT_RATE = 0.1       # Dropout probability
EPOCHS = 15

print(f"Configuration:")
print(f"  Model dimension (d_model): {EMBEDDING_DIM}")
print(f"  Attention heads: {NUM_HEADS}")
print(f"  Head dimension: {EMBEDDING_DIM // NUM_HEADS}")
print(f"  Feed-forward dimension: {FF_DIM}")
print(f"  Encoder/Decoder layers: {NUM_LAYERS}")

Configuration:
  Model dimension (d_model): 128
  Attention heads: 4
  Head dimension: 32
  Feed-forward dimension: 512
  Encoder/Decoder layers: 2


## Step 2: Data Loading and Preprocessing

Same preprocessing as LSTM model - clean text, normalize, add special tokens.

In [3]:
def preprocess_sentence(s):
    """
    Preprocess sentence by lowercasing, handling punctuation, and adding delimiters.
    """
    s = s.lower().strip()
    s = re.sub(r"([?.!,¿])", r" \1 ", s)
    s = re.sub(r'[" "]+', " ", s)
    s = re.sub(r"[^a-zA-Z?.!,¿áéíóúâêôãõçÀÉÍÓÚÂÊÔÃÕÇ]+", " ", s)
    s = s.strip()
    s = "<start> " + s + " <end>"
    return s

def load_dataset(path, num_examples):
    """
    Load English-Portuguese sentence pairs from file.
    """
    lines = open(path, encoding="utf-8").read().strip().split("\n")
    en_sentences, pt_sentences = [], []
    for line in lines[:num_examples]:
        parts = line.split("\t")
        if len(parts) >= 2:
            en_sentences.append(preprocess_sentence(parts[0]))
            pt_sentences.append(preprocess_sentence(parts[1]))
    return en_sentences, pt_sentences

def tokenize(lang):
    """
    Tokenize sentences and pad to uniform length.
    """
    lang_tokenizer = Tokenizer(filters="")
    lang_tokenizer.fit_on_texts(lang)
    tensor = lang_tokenizer.texts_to_sequences(lang)
    tensor = pad_sequences(tensor, padding="post")
    return tensor, lang_tokenizer

print("Loading and preprocessing dataset...")
input_lang, target_lang = load_dataset(FILE_PATH, NUM_EXAMPLES)
print(f"Loaded {len(input_lang)} sentence pairs")

Loading and preprocessing dataset...
Loaded 5000 sentence pairs


In [4]:
# Tokenize both languages
input_tensor, input_tokenizer = tokenize(input_lang)
target_tensor, target_tokenizer = tokenize(target_lang)

# Get dataset statistics
max_length_input = input_tensor.shape[1]
max_length_target = target_tensor.shape[1]
vocab_input_size = len(input_tokenizer.word_index) + 1
vocab_target_size = len(target_tokenizer.word_index) + 1

print(f"English vocabulary: {vocab_input_size}")
print(f"Portuguese vocabulary: {vocab_target_size}")
print(f"Max input length: {max_length_input}")
print(f"Max target length: {max_length_target}")

# Create TensorFlow dataset
dataset = tf.data.Dataset.from_tensor_slices((input_tensor, target_tensor))
dataset = dataset.shuffle(len(input_tensor)).batch(BATCH_SIZE, drop_remainder=True)
print(f"\nDataset batches: {len(dataset)}")

English vocabulary: 1208
Portuguese vocabulary: 2197
Max input length: 8
Max target length: 11

Dataset batches: 78


## Step 3: Positional Encoding

### Why Do We Need Positional Encoding?

Unlike RNNs, Transformers process all positions in parallel. They have no inherent sense of word order!

**Solution**: Add position-dependent patterns (sinusoidal functions) to embeddings:
- Even dimensions: $PE(pos, 2i) = \sin(pos / 10000^{2i/d_{model}})$
- Odd dimensions: $PE(pos, 2i+1) = \cos(pos / 10000^{2i/d_{model}})$

### Advantages:
- Unique encoding for each position
- Encodes relative distances through sin/cos properties
- Generalizes to sequences longer than training data

In [5]:
def positional_encoding(length, depth):
    """
    Generate positional encodings using sinusoidal patterns.
    
    Args:
        length: Maximum sequence length
        depth: Model dimension (d_model)
    
    Returns:
        Tensor of shape (length, depth) with position encodings
    """
    depth = depth // 2
    
    # Position indices: [0, 1, 2, ..., length-1]
    positions = np.arange(length)[:, np.newaxis]  
    
    # Dimension indices divided by depth
    # Creates different frequency bands for each dimension
    depths = np.arange(depth)[np.newaxis, :] / depth  
    
    # Calculate angle rates: 1 / (10000 ^ (depths))
    # Higher dimensions get smaller angle rates (slower oscillations)
    angle_rates = 1 / (10000**depths)  
    
    # Element-wise product: position * angle_rate
    angle_rads = positions * angle_rates  
    
    # Concatenate sin and cos
    # sin for even indices, cos for odd indices
    pos_encoding = np.concatenate([np.sin(angle_rads), np.cos(angle_rads)], axis=-1)
    
    return tf.cast(pos_encoding, dtype=tf.float32)

# Visualize positional encoding
pos_enc = positional_encoding(length=100, depth=128)
print(f"Positional encoding shape: {pos_enc.shape}")
print(f"First position encoding (first 10 dims): {pos_enc[0, :10].numpy()}")
print(f"50th position encoding (first 10 dims): {pos_enc[50, :10].numpy()}")

Positional encoding shape: (100, 128)
First position encoding (first 10 dims): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
50th position encoding (first 10 dims): [-0.26237485 -0.63196105 -0.20298104  0.86898875  0.15662014 -0.70637584
  0.78724194 -0.55706674 -0.10324074  0.90258104]


## Step 4: Positional Embedding Layer

Combines word embedding with positional encoding:
1. Convert word IDs to embeddings
2. Scale embeddings by √d_model (stabilizes training)
3. Add positional encoding to preserve word order information

In [6]:
class PositionalEmbedding(Layer):
    """
    Combines word embedding with positional encoding.
    """
    
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        # Word embedding layer
        self.embedding = Embedding(vocab_size, d_model, mask_zero=False)
        # Pre-compute positional encodings (can handle sequences up to 2048)
        self.pos_encoding = positional_encoding(length=2048, depth=d_model)

    def call(self, x):
        """
        Apply embedding and add positional encoding.
        
        Args:
            x: Input token IDs (batch_size, sequence_length)
        
        Returns:
            Embedded + positioned tokens (batch_size, sequence_length, d_model)
        """
        # Get sequence length from input
        length = tf.shape(x)[1]
        
        # Convert token IDs to embeddings
        x = self.embedding(x)
        
        # Scale embeddings by sqrt(d_model)
        # This scaling helps stabilize training
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        
        # Add positional encoding to the embeddings
        x = x + self.pos_encoding[tf.newaxis, :length, :]
        
        return x

print("PositionalEmbedding layer defined!")

PositionalEmbedding layer defined!


## Step 5: Encoder Layer

### Architecture:
```
Input → Multi-Head Self-Attention → Dropout → Add & Norm ↓
                                                   ↓
              Feed-Forward Network → Dropout → Add & Norm → Output
```

### Components:

**Multi-Head Self-Attention**:
- Multiple parallel attention mechanisms (different representation subspaces)
- Each head: Attention(Q, K, V) = softmax(QK^T/√d_k)V
- Results concatenated and projected

**Feed-Forward Network**:
- FFN(x) = max(0, xW1 + b1)W2 + b2
- Applies same transformation to each position independently

**Residual Connections & Layer Norm**:
- Enable deep networks without vanishing gradients
- Normalize activations for stable training

In [7]:
class EncoderLayer(Layer):
    """
    Single Transformer encoder layer.
    
    Architecture:
    1. Multi-head self-attention
    2. Add & Norm
    3. Feed-forward network
    4. Add & Norm
    """
    
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()
        
        # Multi-head self-attention
        # key_dim = d_model (total dimension across all heads)
        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, 
            key_dim=d_model
        )
        
        # Feed-forward network: d_model → dff → d_model
        self.ffn = tf.keras.Sequential([
            Dense(dff, activation='relu'),
            Dense(d_model)
        ])
        
        # Layer normalization for stability
        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()
        
        # Dropout for regularization
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)

    def call(self, x):
        """
        Process input through encoder layer.
        
        Args:
            x: Input (batch_size, sequence_length, d_model)
        
        Returns:
            Output (batch_size, sequence_length, d_model)
        """
        
        # Self-attention: query, key, value all come from x
        attn_output = self.mha(query=x, value=x, key=x)
        attn_output = self.dropout1(attn_output)
        
        # Residual connection + normalization
        out1 = self.layernorm1(x + attn_output)  
        
        # Feed-forward network
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        
        # Residual connection + normalization
        return self.layernorm2(out1 + ffn_output)

print("EncoderLayer defined!")

EncoderLayer defined!


## Step 6: Decoder Layer

### Architecture:
```
Target → Masked Self-Attention → Dropout → Add & Norm ↓
                                             ↓
         Cross-Attention(with Encoder) → Dropout → Add & Norm ↓
                                             ↓
         Feed-Forward Network → Dropout → Add & Norm → Output
```

### Key Differences from Encoder:

1. **Masked Self-Attention**: Can only attend to previous positions (prevents "cheating" during inference)
2. **Cross-Attention**: Attends to encoder outputs (what encoder learned)
   - Query: Decoder hidden state
   - Key, Value: Encoder outputs
3. **Feed-Forward**: Same as encoder

In [8]:
class DecoderLayer(Layer):
    """
    Single Transformer decoder layer.
    
    Architecture:
    1. Masked multi-head self-attention
    2. Add & Norm
    3. Cross-attention (attends to encoder)
    4. Add & Norm
    5. Feed-forward network
    6. Add & Norm
    """
    
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()
        
        # Masked self-attention (prevents attending to future tokens)
        self.mha1 = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, 
            key_dim=d_model
        )
        
        # Cross-attention (attends to encoder outputs)
        self.mha2 = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, 
            key_dim=d_model
        )
        
        # Feed-forward network
        self.ffn = tf.keras.Sequential([
            Dense(dff, activation='relu'),
            Dense(d_model)
        ])
        
        # Layer normalization
        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()
        self.layernorm3 = LayerNormalization()
        
        # Dropout
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)
        self.dropout3 = Dropout(dropout_rate)

    def call(self, x, enc_output):
        """
        Process input through decoder layer.
        
        Args:
            x: Decoder input (batch_size, target_seq_length, d_model)
            enc_output: Encoder output (batch_size, input_seq_length, d_model)
        
        Returns:
            Output (batch_size, target_seq_length, d_model)
        """
        
        # Masked self-attention: query, key, value from target sequence
        # use_causal_mask=True prevents attending to future positions
        attn1 = self.mha1(query=x, value=x, key=x, use_causal_mask=True)
        attn1 = self.dropout1(attn1)
        out1 = self.layernorm1(x + attn1)
        
        # Cross-attention: query from decoder, key/value from encoder
        # Allows decoder to focus on relevant input parts
        attn2 = self.mha2(query=out1, value=enc_output, key=enc_output)
        attn2 = self.dropout2(attn2)
        out2 = self.layernorm2(out1 + attn2)
        
        # Feed-forward network
        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output)
        
        return self.layernorm3(out2 + ffn_output)

print("DecoderLayer defined!")

DecoderLayer defined!


## Step 7: Full Transformer Model

Combines encoder and decoder stacks:

**Encoder**:
- N stacked encoder layers
- Processes full input sequence
- No masking (can attend to all positions)

**Decoder**:
- N stacked decoder layers
- Processes output sequence
- Masked self-attention (can only attend to previous positions)
- Cross-attention to encoder outputs

**Output**:
- Dense layer projects to vocabulary size
- Outputs logits for each token in vocabulary

In [9]:
class Transformer(tf.keras.Model):
    """
    Full Transformer model for sequence-to-sequence tasks.
    """
    
    def __init__(self, num_layers, d_model, num_heads, dff, 
                 input_vocab_size, target_vocab_size, dropout_rate=0.1):
        super().__init__()
        
        # Encoder embedding
        self.enc_embedding = PositionalEmbedding(
            vocab_size=input_vocab_size, 
            d_model=d_model
        )
        
        # Decoder embedding
        self.dec_embedding = PositionalEmbedding(
            vocab_size=target_vocab_size, 
            d_model=d_model
        )
        
        # Stack of encoder layers
        self.encoder_layers = [
            EncoderLayer(d_model, num_heads, dff, dropout_rate) 
            for _ in range(num_layers)
        ]
        
        # Stack of decoder layers
        self.decoder_layers = [
            DecoderLayer(d_model, num_heads, dff, dropout_rate) 
            for _ in range(num_layers)
        ]
        
        # Final output projection to vocabulary
        self.final_layer = Dense(target_vocab_size)
        
        # Dropout for regularization
        self.dropout = Dropout(dropout_rate)

    def call(self, inputs):
        """
        Forward pass through transformer.
        
        Args:
            inputs: Tuple of (encoder_input, decoder_input)
                   encoder_input shape: (batch_size, input_length)
                   decoder_input shape: (batch_size, target_length)
        
        Returns:
            logits: (batch_size, target_length, target_vocab_size)
        """
        inp, tar = inputs

        # ===== ENCODER =====
        # Embed and add position information
        x = self.enc_embedding(inp)
        x = self.dropout(x)
        
        # Pass through encoder layers
        for layer in self.encoder_layers:
            x = layer(x)
        enc_output = x

        # ===== DECODER =====
        # Embed target and add position information
        y = self.dec_embedding(tar)
        y = self.dropout(y)
        
        # Pass through decoder layers (with encoder output for cross-attention)
        for layer in self.decoder_layers:
            y = layer(y, enc_output)
        
        # Project to vocabulary size
        logits = self.final_layer(y)
        
        return logits

print("Transformer model defined!")

Transformer model defined!


## Step 8: Initialize Model

In [10]:
# Initialize Transformer
transformer = Transformer(
    num_layers=NUM_LAYERS,
    d_model=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    dff=FF_DIM,
    input_vocab_size=vocab_input_size,
    target_vocab_size=vocab_target_size,
    dropout_rate=DROPOUT_RATE
)

print(f"Transformer initialized:")
print(f"  Model dimension: {EMBEDDING_DIM}")
print(f"  Attention heads: {NUM_HEADS} (dimension per head: {EMBEDDING_DIM // NUM_HEADS})")
print(f"  Feed-forward dimension: {FF_DIM}")
print(f"  Encoder layers: {NUM_LAYERS}")
print(f"  Decoder layers: {NUM_LAYERS}")
print(f"  Total trainable parameters: {sum([tf.size(w).numpy() for w in transformer.trainable_variables]):,}")

Transformer initialized:
  Model dimension: 128
  Attention heads: 4 (dimension per head: 32)
  Feed-forward dimension: 512
  Encoder layers: 2
  Decoder layers: 2
  Total trainable parameters: 0


## Step 9: Training Setup

### Teacher Forcing in Transformers:

Unlike RNNs that decode sequentially, Transformers can process entire target sequences in parallel during training.

**Training**: 
- Input target sequence with special handling
- `tar_inp` = target[:-1] (all tokens except <end>)
- `tar_real` = target[1:] (all tokens except <start>)
- This shift ensures masking prevents the model from "seeing" future tokens it should predict

**Inference**:
- Autoregressive: Generate one token at a time
- Feed generated tokens back as input

### Masked Loss:
- Ignore padding tokens (token ID = 0) in loss computation
- Only penalize predictions on real tokens

In [11]:
# ==========================================
# TRAINING SETUP WITH MASKED LOSS
# ==========================================

# Adam optimizer with lower learning rate (Transformers are sensitive to LR)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

# Sparse categorical cross-entropy (targets are integer indices)
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,  # Model outputs logits, not probabilities
    reduction='none'   # Don't reduce - we'll mask manually
)

def loss_function(real, pred):
    """
    Compute loss while masking padding tokens.
    
    Args:
        real: True target tokens (batch_size, vocab_size)
        pred: Predicted logits (batch_size, vocab_size)
    
    Returns:
        float: Masked loss
    """
    # Create mask for non-padding tokens
    mask = tf.math.not_equal(real, 0)
    
    # Compute cross-entropy for all tokens
    loss_ = loss_object(real, pred)
    
    # Apply mask: set padding token losses to 0
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    
    # Average loss over non-padding tokens
    return tf.reduce_sum(loss_) / tf.reduce_sum(mask)

print("Loss function and optimizer configured!")

Loss function and optimizer configured!


## Step 10: Training Loop with Teacher Forcing

### Why Teacher Forcing Works Well in Transformers:

1. **Parallel Processing**: Can process entire target sequence at once
2. **Causal Masking**: Masking ensures model can't "see" future tokens
3. **Efficiency**: Training much faster than RNN autoregressive decoding
4. **Stability**: Gradients flow directly from all positions

In [12]:
@tf.function
def train_step(inp, targ):
    """
    Execute one training step (one batch).
    
    Args:
        inp: Input sequences (batch_size, input_length)
        targ: Target sequences (batch_size, target_length)
    
    Returns:
        float: Loss for this batch
    """
    # Shift target: decoder input is all tokens except <end>
    tar_inp = targ[:, :-1]
    
    # Target labels: all tokens except <start>
    tar_real = targ[:, 1:]

    with tf.GradientTape() as tape:
        # Forward pass through transformer
        # Model receives both encoder input and shifted decoder input
        predictions = transformer([inp, tar_inp], training=True)
        
        # Compute masked loss
        loss = loss_function(tar_real, predictions)

    # Backpropagation
    gradients = tape.gradient(loss, transformer.trainable_variables)
    optimizer.apply_gradients(zip(gradients, transformer.trainable_variables))
    
    return loss

print("Training function defined!")

Training function defined!


## Step 11: Execute Training

Train the Transformer model for the specified number of epochs.

In [13]:
print("\nStarting Transformer Training...\n")

for epoch in range(EPOCHS):
    total_loss = 0
    batch_count = 0
    
    # Iterate through all batches
    for batch, (inp, targ) in enumerate(dataset):
        batch_loss = train_step(inp, targ)
        total_loss += batch_loss
        batch_count = batch

        # Print progress every 100 batches
        if batch % 100 == 0:
            print(f"Epoch {epoch + 1} Batch {batch} Loss {batch_loss.numpy():.4f}")

    # Print epoch summary
    avg_loss = total_loss / (batch_count + 1)
    print(f"Epoch {epoch + 1} Complete. Average Loss: {avg_loss:.4f}\n")

print("Training completed!")


Starting Transformer Training...

Epoch 1 Batch 0 Loss 7.8989
Epoch 1 Complete. Average Loss: 4.9559

Epoch 2 Batch 0 Loss 3.8373
Epoch 2 Complete. Average Loss: 3.4793

Epoch 3 Batch 0 Loss 3.1473
Epoch 3 Complete. Average Loss: 2.8422

Epoch 4 Batch 0 Loss 2.4166
Epoch 4 Complete. Average Loss: 2.4058

Epoch 5 Batch 0 Loss 2.2396
Epoch 5 Complete. Average Loss: 2.0899

Epoch 6 Batch 0 Loss 1.8605
Epoch 6 Complete. Average Loss: 1.8358

Epoch 7 Batch 0 Loss 1.8349
Epoch 7 Complete. Average Loss: 1.6241

Epoch 8 Batch 0 Loss 1.3781
Epoch 8 Complete. Average Loss: 1.4318

Epoch 9 Batch 0 Loss 1.3076
Epoch 9 Complete. Average Loss: 1.2817

Epoch 10 Batch 0 Loss 1.1570
Epoch 10 Complete. Average Loss: 1.1388

Epoch 11 Batch 0 Loss 0.9812
Epoch 11 Complete. Average Loss: 1.0162

Epoch 12 Batch 0 Loss 0.9471
Epoch 12 Complete. Average Loss: 0.9118

Epoch 13 Batch 0 Loss 0.7748
Epoch 13 Complete. Average Loss: 0.8198

Epoch 14 Batch 0 Loss 0.7463
Epoch 14 Complete. Average Loss: 0.7389

Epo

## Step 12: Inference - Autoregressive Decoding

### Key Differences from Training:

1. **No Teacher Forcing**: Model doesn't know true target
2. **Sequential Generation**: Generate one token at a time
3. **Greedy Decoding**: Select highest probability token (simple but not optimal)
   - Alternative: Beam search for better translations

### Process:
1. Encode entire input sequence (done once)
2. Start decoder with `<start>` token
3. Generate one token: pass current sequence through decoder
4. Select highest probability token
5. Append to sequence and repeat until `<end>` or max length

In [14]:
def translate(sentence):
    """
    Translate an English sentence to Portuguese using the trained Transformer.
    
    Args:
        sentence (str): English sentence to translate
    """
    # Preprocess input sentence
    cleaned_sentence = preprocess_sentence(sentence)
    
    # Tokenize: convert words to indices
    inputs = [
        input_tokenizer.word_index[i] 
        for i in cleaned_sentence.split(' ') 
        if i in input_tokenizer.word_index
    ]
    
    # Pad to encoder's expected length
    inputs = pad_sequences([inputs], maxlen=max_length_input, padding='post')
    encoder_input = tf.convert_to_tensor(inputs)

    # Token indices
    start_index = target_tokenizer.word_index['<start>']
    end_index = target_tokenizer.word_index['<end>']
    
    # Initialize decoder output with start token
    output_tokens = tf.convert_to_tensor([[start_index]], dtype=tf.int32)

    # Autoregressive decoding: generate one token at a time
    for _ in range(max_length_target):
        # Forward pass through transformer
        # Encoder input stays same, decoder input grows
        predictions = transformer([encoder_input, output_tokens], training=False)
        
        # Get prediction for last position (current step)
        prediction = predictions[:, -1, :]
        
        # Greedy decoding: select token with highest probability
        predicted_id = tf.argmax(prediction, axis=-1, output_type=tf.int32)[0]

        # Stop if end token is generated
        if predicted_id == end_index:
            break

        # Append predicted token to output sequence
        output_tokens = tf.concat(
            [output_tokens, tf.expand_dims([predicted_id], 0)], 
            axis=-1
        )

    # Convert token indices back to words
    translated_tokens = output_tokens.numpy()[0][1:]  # Skip <start> token
    result = " ".join([
        target_tokenizer.index_word[i] 
        for i in translated_tokens 
        if i in target_tokenizer.index_word
    ])
    
    print(f"Input: {sentence}")
    print(f"Predicted translation: {result}\n")

print("Translation function defined!")

Translation function defined!


## Step 13: Test Translation

Test the trained Transformer on some example sentences.

In [15]:
print("--- Testing Transformer Translation Capabilities ---\n")

test_sentences = [
    "Go.",
    "Run!",
    "I love studying languages."
]

for sent in test_sentences:
    translate(sent)

--- Testing Transformer Translation Capabilities ---

Input: Go.
Predicted translation: vá .

Input: Run!
Predicted translation: corra !

Input: I love studying languages.
Predicted translation: eu amo o .



## Summary: Transformer vs LSTM with Attention

### Transformer Advantages:

| Aspect | LSTM + Attention | Transformer |
|--------|------------------|-------------|
| **Processing** | Sequential (step-by-step) | Parallel (all positions) |
| **Speed** | Slower (can't parallelize) | Faster training |
| **Long-range Dependencies** | Good | Better (direct attention) |
| **Training Stability** | Can have vanishing gradients | More stable (residual + norm) |
| **Memory Usage** | Lower | Higher (stores attention matrices) |
| **Scalability** | Limited | Excellent (scales to 100B+ parameters) |

### Key Insights:

1. **Positional Encoding**: Gives Transformers sense of word order
2. **Multi-Head Attention**: Different representation subspaces
3. **Residual Connections**: Enable deep networks (many layers)
4. **Layer Normalization**: Stabilizes training
5. **Causal Masking**: Prevents cheating during training
6. **Cross-Attention**: Allows decoder to focus on input

### Transformer Variants:

- **BERT**: Encoder-only (masked language modeling)
- **GPT**: Decoder-only (autoregressive language generation)
- **T5**: Encoder-decoder (what we built)

### Next Steps:

- Implement **Beam Search** for better translations
- Add **Warmup and Learning Rate Scheduling**
- Try **Byte-Pair Encoding (BPE)** for subword tokenization
- Visualize **Attention Weights** to understand model decisions
- Experiment with different hyperparameters
- Train on larger datasets with more layers